# ATD vs ATD_2 Comparison

`ATD` (existing) = `order_final_state_timestamp_local` - `restaurant_offered_timestamp_utc` (adjusted to UTC)

`ATD_2` (new) = `order_final_state_timestamp_local` - `eater_request_timestamp_local`

ATD_2 captures total time from customer request to delivery, while ATD starts the clock when the restaurant receives the order.

In [ ]:
import pandas as pd
from pathlib import Path

RAW_PATH = Path('../data/raw/BC_A&A_with_ATD.csv')

df = pd.read_csv(
    RAW_PATH,
    usecols=[
        'delivery_trip_uuid',
        'workflow_uuid',
        'order_final_state_timestamp_local',
        'eater_request_timestamp_local',
        'restaurant_offered_timestamp_utc',
        'ATD',
    ],
    parse_dates=[
        'order_final_state_timestamp_local',
        'eater_request_timestamp_local',
        'restaurant_offered_timestamp_utc',
    ],
    na_values=['\\N', 'NULL', 'None', ''],
)

print(f'Rows loaded: {len(df):,}')
df.head(3)

In [ ]:
df = df.dropna(subset=[
    'order_final_state_timestamp_local',
    'eater_request_timestamp_local',
    'ATD',
])

# ATD_2: both timestamps are local, no conversion needed
df['ATD_2'] = (
    df['order_final_state_timestamp_local'] - df['eater_request_timestamp_local']
).dt.total_seconds() / 60

df['ATD_diff'] = df['ATD_2'] - df['ATD']

print(f'Rows after dropping nulls: {len(df):,}')
df[['ATD', 'ATD_2', 'ATD_diff']].head(5)

In [ ]:
summary = pd.DataFrame({
    'ATD':      df['ATD'].describe(percentiles=[.05, .25, .5, .75, .95, .99]),
    'ATD_2':    df['ATD_2'].describe(percentiles=[.05, .25, .5, .75, .95, .99]),
    'ATD_diff': df['ATD_diff'].describe(percentiles=[.05, .25, .5, .75, .95, .99]),
}).round(4)

print(summary.to_string())
print(f'\nMean ATD_diff (ATD_2 - ATD) : {df["ATD_diff"].mean():.4f} min')
print(f'Rows where ATD_2 > ATD      : {(df["ATD_diff"] > 0).sum():,}  ({(df["ATD_diff"] > 0).mean()*100:.2f}%)')
print(f'Rows where ATD_2 < ATD      : {(df["ATD_diff"] < 0).sum():,}  ({(df["ATD_diff"] < 0).mean()*100:.2f}%)')
print(f'Rows where ATD_2 == ATD     : {(df["ATD_diff"] == 0).sum():,}  ({(df["ATD_diff"] == 0).mean()*100:.2f}%)')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# ATD vs ATD_2 distributions overlaid
p99 = max(df['ATD'].quantile(0.99), df['ATD_2'].quantile(0.99))
axes[0].hist(df['ATD'].clip(0, p99), bins=80, alpha=0.6,
             color='steelblue', edgecolor='none', label='ATD')
axes[0].hist(df['ATD_2'].clip(0, p99), bins=80, alpha=0.6,
             color='coral', edgecolor='none', label='ATD_2')
axes[0].set_title('ATD vs ATD_2 distribution (0 to p99)')
axes[0].set_xlabel('minutes')
axes[0].legend()

# Scatter sample
sample = df.sample(min(20_000, len(df)), random_state=42)
axes[1].scatter(sample['ATD'], sample['ATD_2'], alpha=0.05, s=3, color='teal')
lim = max(sample['ATD'].quantile(0.99), sample['ATD_2'].quantile(0.99))
axes[1].plot([0, lim], [0, lim], 'r--', linewidth=1)
axes[1].set_xlabel('ATD (min)')
axes[1].set_ylabel('ATD_2 (min)')
axes[1].set_title('ATD vs ATD_2 scatter (sample 20k)')

# Difference distribution
diff_clipped = df['ATD_diff'].clip(
    df['ATD_diff'].quantile(0.01),
    df['ATD_diff'].quantile(0.99),
)
axes[2].hist(diff_clipped, bins=80, color='mediumpurple', edgecolor='none')
axes[2].axvline(df['ATD_diff'].mean(), color='red', linewidth=1.5,
                label=f'mean = {df["ATD_diff"].mean():.2f}')
axes[2].axvline(0, color='black', linewidth=1, linestyle='--')
axes[2].set_title('ATD_diff = ATD_2 - ATD (p1-p99 clipped)')
axes[2].set_xlabel('minutes')
axes[2].legend()

plt.tight_layout()
plt.show()